# SpaceX Falcon 9 – Launch Site Locations with Folium
Mark launch sites, show success and failure per site, and measure distances to nearby features.

Open the saved HTML maps in `maps/` in a browser to explore them interactively.

In [1]:
import folium
import pandas as pd
import os
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians
os.makedirs("maps", exist_ok=True)
spacex_df = pd.read_csv("data/dataset_part_2.csv")[['LaunchSite', 'Latitude', 'Longitude', 'Class']]

## Task 1: Mark all launch sites on a map

In [2]:
launch_sites_df = spacex_df.groupby('LaunchSite', as_index=False).first()[['LaunchSite', 'Latitude', 'Longitude']]
launch_sites_df

,LaunchSite,Latitude,Longitude
0,CCAFS SLC 40,28.561857,-80.577366
1,KSC LC 39A,28.608058,-80.603956
2,VAFB SLC 4E,34.632093,-120.610829


In [3]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=[31, -100], zoom_start=4)
for _, r in launch_sites_df.iterrows():
    coord = [r['Latitude'], r['Longitude']]
    folium.Circle(coord, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(r['LaunchSite'])).add_to(site_map)
    folium.map.Marker(coord, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % r['LaunchSite'])).add_to(site_map)
site_map.save("maps/1_launch_sites.html")
site_map

All sites are on the coast. The two Florida sites sit next to each other on Cape Canaveral; VAFB SLC 4E is on the California coast for polar launches.

## Task 2: Mark the success and failed launches for each site

In [4]:
marker_cluster = MarkerCluster()
spacex_df['marker_color'] = spacex_df['Class'].map({1: 'green', 0: 'red'})
outcome_map = folium.Map(location=[31, -100], zoom_start=4)
outcome_map.add_child(marker_cluster)
for _, r in spacex_df.iterrows():
    folium.Marker([r['Latitude'], r['Longitude']],
        icon=folium.Icon(color='white', icon_color=r['marker_color']),
        popup=f"{r['LaunchSite']}: {'Success' if r['Class'] else 'Failure'}").add_to(marker_cluster)
outcome_map.save("maps/2_launch_outcomes.html")
outcome_map

In [5]:
spacex_df.groupby('LaunchSite')['Class'].agg(['sum', 'count', 'mean'])

,sum,count,mean
LaunchSite,,,
CCAFS SLC 40,33,55,0.600000
KSC LC 39A,17,22,0.772727
VAFB SLC 4E,10,13,0.769231


KSC LC 39A has the highest success rate (77.3%), followed by VAFB SLC 4E (76.9%) and CCAFS SLC 40 (60.0%).

## Task 3: Calculate distances between a launch site and its proximities

In [6]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

In [7]:
# CCAFS SLC 40, and points read off the map with MousePosition
site = (28.56230197, -80.57735648)
points = {
    "Coastline":    (28.56367, -80.56794),
    "Railway":      (28.57208, -80.58528),
    "Highway":      (28.56335, -80.57086),
    "City (Cape Canaveral)": (28.40577, -80.60476),
}
prox_map = folium.Map(location=site, zoom_start=14)
prox_map.add_child(MousePosition(position='topright', separator=' Long: ', prefix='Lat:', num_digits=20))
folium.Marker(site, popup="CCAFS SLC 40").add_to(prox_map)
rows = []
for name, pt in points.items():
    d = calculate_distance(*site, *pt)
    rows.append((name, round(d, 2)))
    folium.Marker(pt, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s: %.2f KM</b></div>' % (name, d))).add_to(prox_map)
    folium.PolyLine(locations=[site, pt], weight=2).add_to(prox_map)
prox_map.save("maps/3_proximities.html")
pd.DataFrame(rows, columns=["Feature", "Distance (km)"])

,Feature,Distance (km)
0,Coastline,0.93
1,Railway,1.33
2,Highway,0.65
3,City (Cape Canaveral),17.62


In [8]:
prox_map

Launch sites are within about 1 km of the coastline, highway and railway, but well away from towns. Coast access lets spent stages fall over water; road and rail bring in heavy hardware; distance from cities keeps the public safe.